# GO terms per YeastMap complex: true complex vs. true-only vs. added proteins

Per YeastMap complex (`input_12_yeastmap.tsv`) three protein groups:
- **true_complex**: all proteins of the matched Complex Portal complex (`true_complex`)
- **true_only**: proteins of the true complex that are not in the prediction (`true_complex` − `predicted_complex`)
- **added**: proteins in the prediction that are not in the true complex (`predicted_complex` − `true_complex`)

For each group: the GO terms (CC, BP, MF) its proteins are annotated with, and whether the term also occurs in the true complex.

In [2]:
import gzip
import urllib.request

import polars as pl
from goatools.obo_parser import GODag
from procompa import get_project_root

DATA = get_project_root() / "data"
YEASTMAP_TSV = DATA / "Pipeline/12_Yeastmap/input_12_yeastmap.tsv"
GO_DIR = DATA / "GO"
DOWNLOADS = {   # file name -> url (only downloaded if the file is missing)
    "goa_yeast.gaf.gz": "https://ftp.ebi.ac.uk/pub/databases/GO/goa/YEAST/goa_yeast.gaf.gz",
    "go-basic.obo": "https://purl.obolibrary.org/obo/go/go-basic.obo",  
}
EXCLUDED_EVIDENCE = ["IPI"]   
ASPECT_NAMES = {"C": "cellular_component", "P": "biological_process", "F": "molecular_function"}

assert YEASTMAP_TSV.exists(), f"missing: {YEASTMAP_TSV}"

## 1) Download Complex Portal (yeast), GOA yeast annotations and the GO ontology

In [5]:
import shutil

GO_DIR.mkdir(parents=True, exist_ok=True)
for file_name, url in DOWNLOADS.items():
    target_path = GO_DIR / file_name
    if not target_path.exists():
        print(f"downloading {url}")
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})   # default Python agent gets 403
        with urllib.request.urlopen(request) as response, open(target_path, "wb") as target_file:
            shutil.copyfileobj(response, target_file)
    print(f"{file_name}: {target_path.stat().st_size / 1e6:.1f} MB")

goa_yeast.gaf.gz: 2.1 MB
downloading https://purl.obolibrary.org/obo/go/go-basic.obo
go-basic.obo: 32.2 MB


## 2) shared / added / missing proteins per YeastMap complex

In [ ]:
yeastmap = pl.read_csv(YEASTMAP_TSV, separator="\t", infer_schema_length=0)
assert yeastmap["#Complex ac"].is_unique().all(), "YeastMap complex listed twice"

protein_groups = (
    yeastmap
    .select(
        complex_ac="#Complex ac",
        cp_complex_ac="#Complex_ac_db",
        true_complex=pl.col("true_complex").str.split(" "),
        predicted=pl.col("predicted_complex").str.split(" "),
    )
    .with_columns(
        true_only=pl.col("true_complex").list.set_difference("predicted"),
        added=pl.col("predicted").list.set_difference("true_complex"),
    )
)

# long format: one row per (complex, group, protein)
group_members = pl.concat([
    protein_groups.select("complex_ac", "cp_complex_ac", pl.lit(group).alias("group"), pl.col(group).alias("protein"))
    .explode("protein").drop_nulls("protein")
    for group in ("true_complex", "true_only", "added")
])
group_members.group_by("group").agg(n_proteins=pl.len(), n_complexes=pl.col("complex_ac").n_unique())

group,n_proteins,n_complexes
str,u32,u32
"""true_complex""",533,156
"""true_only""",90,49
"""added""",533,156


## 2) GO annotations (direct terms; without `NOT` and IPI)

In [ ]:
go_dag = GODag(str(GO_DIR / "go-basic.obo"), prt=None)

# go_annotations of entire yeats proteome
go_annotations = (
    pl.read_csv(gzip.open(GO_DIR / "goa_yeast.gaf.gz").read(), separator="\t", comment_prefix="!",
                has_header=False, quote_char=None, infer_schema_length=0)
    .select(pl.nth(0).alias("db"), pl.nth(1).alias("protein"), pl.nth(3).alias("qualifier"),   # GAF columns 1, 2, 4, 5, 7, 9
            pl.nth(4).alias("go_id"), pl.nth(6).alias("evidence"), pl.nth(8).alias("aspect"))
    .filter(
        (pl.col("db") == "UniProtKB")
        & ~pl.col("qualifier").fill_null("").str.contains("NOT")
        & ~pl.col("evidence").is_in(EXCLUDED_EVIDENCE)
    )
    .select("protein", "go_id", "aspect")
    .unique()
)
assert go_annotations["aspect"].is_in(list(ASPECT_NAMES)).all(), "unexpected aspect in GAF"

unknown_term_ids = set(go_annotations["go_id"]) - set(go_dag)
print(f"{go_annotations.height} annotations of {go_annotations['protein'].n_unique()} proteins; "
      f"{len(unknown_term_ids)} term ids not in go-basic.obo (obsolete) -> shown without name")
go_term_names = pl.DataFrame({"go_id": list(go_dag), "go_name": [go_dag[go_id].name for go_id in go_dag]})

59007 annotations of 6056 proteins; 0 term ids not in go-basic.obo (obsolete) -> shown without name


## 3) GO terms per complex and group
`in_true_complex`: the term is also annotated to at least one protein of the true complex.

In [ ]:
group_go_terms = (
    group_members
    .join(go_annotations, on="protein", how="inner")
    .group_by("complex_ac", "cp_complex_ac", "group", "aspect", "go_id")
    .agg(n_proteins=pl.len(), proteins=pl.col("protein").sort())
    .join(go_term_names, on="go_id", how="left")
)
true_complex_terms = (group_go_terms.filter(pl.col("group") == "true_complex")
                      .select("complex_ac", "go_id", in_true_complex=pl.lit(True)))
group_go_terms = (
    group_go_terms
    .join(true_complex_terms, on=["complex_ac", "go_id"], how="left")
    .with_columns(pl.col("in_true_complex").fill_null(False),
                  aspect=pl.col("aspect").replace_strict(ASPECT_NAMES))
    .sort("complex_ac", "group", "aspect", "n_proteins", descending=[False, False, False, True])
)
group_go_terms.head()
#group_go_terms.write_parquet(GO_DIR/ "ym_protein_mapping.parquet")

complex_ac,cp_complex_ac,group,aspect,go_id,n_proteins,proteins,go_name,in_true_complex
str,str,str,str,str,u32,list[str],str,bool
"""pred_CPX_1""","""CPX-599""","""added""","""biological_process""","""GO:0034475""",1,"[""P38801""]","""U4 snRNA 3'-end processing""",true
"""pred_CPX_1""","""CPX-599""","""added""","""biological_process""","""GO:0000956""",1,"[""Q08491""]","""nuclear-transcribed mRNA catab…",true
"""pred_CPX_1""","""CPX-599""","""added""","""biological_process""","""GO:0070481""",1,"[""Q08491""]","""nuclear-transcribed mRNA catab…",true
"""pred_CPX_1""","""CPX-599""","""added""","""biological_process""","""GO:0010468""",1,"[""P38801""]","""regulation of gene expression""",false
"""pred_CPX_1""","""CPX-599""","""added""","""biological_process""","""GO:0000973""",1,"[""P38801""]","""post-transcriptional tethering…",true


## 4) Summary per complex
Per group: how many proteins have GO annotations, and for **added** proteins how many share at least one
GO term with the true complex (per aspect).

In [16]:
annotated_per_group = (
    group_members.join(go_annotations.select("protein").unique(), on="protein", how="semi")
    .group_by("complex_ac", "group").agg(n_annotated=pl.len())
)
group_sizes = group_members.group_by("complex_ac", "group").agg(n_proteins=pl.len())

added_sharing_true_terms = (
    group_go_terms.filter((pl.col("group") == "added") & pl.col("in_true_complex"))
    .explode("proteins")
    .group_by("complex_ac", "aspect").agg(n_added_sharing=pl.col("proteins").n_unique())
    .pivot(on="aspect", index="complex_ac", values="n_added_sharing")
    .rename(lambda column: column if column == "complex_ac" else f"added_sharing_{column}")
)

complex_go_summary = (
    group_sizes.join(annotated_per_group, on=["complex_ac", "group"], how="left")
    .with_columns(pl.col("n_annotated").fill_null(0))
    .pivot(on="group", index="complex_ac", values=["n_proteins", "n_annotated"])
    .join(added_sharing_true_terms, on="complex_ac", how="left")
    .with_columns(pl.col("^added_sharing_.*$").fill_null(0))     # no shared term -> 0, not null
    .join(protein_groups.select("complex_ac", "cp_complex_ac"), on="complex_ac")
    .sort("complex_ac")
)
complex_go_summary

complex_ac,n_proteins_true_complex,n_proteins_true_only,n_proteins_added,n_annotated_true_complex,n_annotated_true_only,n_annotated_added,added_sharing_cellular_component,added_sharing_molecular_function,added_sharing_biological_process,cp_complex_ac
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,str
"""pred_CPX_1""",11,null,2,11,null,2,2,1,2,"""CPX-599"""
"""pred_CPX_165""",8,1,4,8,1,4,4,1,4,"""CPX-1885"""
"""pred_CPX_17""",3,null,5,3,null,5,5,3,4,"""CPX-1682"""
"""pred_CPX_174""",11,null,7,11,null,7,7,5,7,"""CPX-599"""
"""pred_CPX_175""",16,null,2,16,null,2,2,2,2,"""CPX-1888"""
…,…,…,…,…,…,…,…,…,…,…
"""pred_CPX_732""",5,null,5,5,null,5,5,3,3,"""CPX-429"""
"""pred_CPX_734""",4,null,6,4,null,6,3,2,1,"""CPX-737"""
"""pred_CPX_735""",2,1,9,2,1,9,8,0,0,"""CPX-1808"""


In [17]:
EXAMPLE_COMPLEX = "pred_CPX_720"
example_terms = group_go_terms.filter(pl.col("complex_ac") == EXAMPLE_COMPLEX)
for group in ("true_complex", "true_only", "added"):
    print(f"\n── {group} ──")
    display(example_terms.filter(pl.col("group") == group)
            .select("aspect", "go_id", "go_name", "n_proteins", "proteins", "in_true_complex"))


── true_complex ──


aspect,go_id,go_name,n_proteins,proteins,in_true_complex
str,str,str,u32,list[str],bool
"""biological_process""","""GO:1902600""","""proton transmembrane transport""",10,"[""P00127"", ""P00128"", … ""P37299""]",true
"""biological_process""","""GO:0045333""","""cellular respiration""",10,"[""P00127"", ""P00128"", … ""P37299""]",true
"""biological_process""","""GO:0006122""","""mitochondrial electron transpo…",10,"[""P00127"", ""P00128"", … ""P37299""]",true
"""biological_process""","""GO:0009060""","""aerobic respiration""",9,"[""P00127"", ""P00128"", … ""P37299""]",true
"""biological_process""","""GO:0034551""","""mitochondrial respiratory chai…",2,"[""P00128"", ""P22289""]",true
…,…,…,…,…,…
"""molecular_function""","""GO:0004222""","""metalloendopeptidase activity""",2,"[""P07256"", ""P07257""]",true
"""molecular_function""","""GO:0009055""","""electron transfer activity""",2,"[""P00163"", ""P07143""]",true
"""molecular_function""","""GO:0016491""","""oxidoreductase activity""",2,"[""P00163"", ""P08067""]",true



── true_only ──


aspect,go_id,go_name,n_proteins,proteins,in_true_complex
str,str,str,u32,list[str],bool
"""biological_process""","""GO:0006122""","""mitochondrial electron transpo…",6,"[""P00127"", ""P00163"", … ""P37299""]",true
"""biological_process""","""GO:1902600""","""proton transmembrane transport""",6,"[""P00127"", ""P00163"", … ""P37299""]",true
"""biological_process""","""GO:0045333""","""cellular respiration""",6,"[""P00127"", ""P00163"", … ""P37299""]",true
"""biological_process""","""GO:0009060""","""aerobic respiration""",5,"[""P00127"", ""P00163"", … ""P37299""]",true
"""biological_process""","""GO:0022904""","""respiratory electron transport…",1,"[""P00163""]",true
…,…,…,…,…,…
"""cellular_component""","""GO:0034399""","""nuclear periphery""",1,"[""P00127""]",true
"""molecular_function""","""GO:0008121""","""quinol-cytochrome-c reductase …",6,"[""P00127"", ""P00163"", … ""P37299""]",true
"""molecular_function""","""GO:0009055""","""electron transfer activity""",2,"[""P00163"", ""P07143""]",true



── added ──


aspect,go_id,go_name,n_proteins,proteins,in_true_complex
str,str,str,u32,list[str],bool
"""biological_process""","""GO:1902600""","""proton transmembrane transport""",5,"[""P00424"", ""P00427"", … ""Q01519""]",true
"""biological_process""","""GO:0006123""","""mitochondrial electron transpo…",5,"[""P00424"", ""P00427"", … ""Q01519""]",false
"""biological_process""","""GO:0006119""","""oxidative phosphorylation""",5,"[""P00424"", ""P00427"", … ""Q01519""]",false
"""biological_process""","""GO:0033617""","""mitochondrial respiratory chai…",3,"[""P04037"", ""P53721"", ""Q01519""]",false
"""biological_process""","""GO:1903457""","""lactate catabolic process""",1,"[""P32891""]",false
…,…,…,…,…,…
"""molecular_function""","""GO:0008720""","""D-lactate dehydrogenase (NAD+)…",1,"[""P32891""]",false
"""molecular_function""","""GO:0003824""","""catalytic activity""",1,"[""P32891""]",false
"""molecular_function""","""GO:0050660""","""flavin adenine dinucleotide bi…",1,"[""P32891""]",false
